# Toy Dataset Analysis — LSA COPIDIS/GCBA
Visualización y análisis de los 29 videos curados para Signformer/LSA-T.

In [ ]:
import json, yaml, re, warnings
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter
warnings.filterwarnings('ignore')

try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
    HAS_WIDGETS = True
except ImportError:
    HAS_WIDGETS = False
    print('ipywidgets no disponible — se usará visualización estática')

%matplotlib inline
plt.rcParams.update({'figure.dpi': 100, 'axes.spines.top': False, 'axes.spines.right': False})

In [ ]:
REPO_ROOT = Path('..').resolve()

with open(REPO_ROOT / 'config.yaml') as f:
    cfg = yaml.safe_load(f)

VIDEOS_DIR = REPO_ROOT / cfg['paths']['videos']
DATASET_DIR = REPO_ROOT / 'data' / 'dataset'
TOY_TEST_PATH  = DATASET_DIR / 'toy_test.json'
TOY_DATASET_PATH = DATASET_DIR / 'toy_dataset.json'

SAMPLE_RATE = 2  # valor usado en build_toy_dataset.py

print(f'Videos dir : {VIDEOS_DIR}')
print(f'toy_test   : {TOY_TEST_PATH.exists()}')
print(f'toy_dataset: {TOY_DATASET_PATH.exists()}')

## Sección 1 — Visualización de keypoints sobre frame real
Slider interactivo sobre el primer video disponible (`toy_test.json`).

In [ ]:
with open(TOY_TEST_PATH) as f:
    example_data = json.load(f)

ex = example_data[0]
print(f"ID       : {ex['id']}")
print(f"Text     : {ex['text'][:90]}...")
print(f"Frames   : {ex['n_frames']}")
print(f"Intent   : {ex['metadata']['intent']}")
print(f"Conf avg : {ex['metadata']['confidence_avg']:.3f}")

In [ ]:
def extract_kps(vector):
    v = np.array(vector)
    return (
        v[0:66].reshape(33, 2),
        v[66:1002].reshape(468, 2),
        v[1002:1044].reshape(21, 2),
        v[1044:1086].reshape(21, 2),
    )

def figsize_for_frame(img, max_w=7):
    h, w = img.shape[:2]
    scale = max_w / w
    return (max_w, h * scale)

def draw_kps(ax, img_bgr, vector):
    h, w = img_bgr.shape[:2]
    pose, face, lh, rh = extract_kps(vector)
    ax.imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))

    face_px = face * [w, h]
    ax.scatter(face_px[:,0], face_px[:,1], s=0.4, c='cyan', alpha=0.4)

    pose_px = pose * [w, h]
    ax.scatter(pose_px[:,0], pose_px[:,1], s=12, c='lime', zorder=5)

    if lh.any():
        lh_px = lh * [w, h]
        ax.scatter(lh_px[:,0], lh_px[:,1], s=10, c='dodgerblue', zorder=5)

    if rh.any():
        rh_px = rh * [w, h]
        ax.scatter(rh_px[:,0], rh_px[:,1], s=10, c='tomato', zorder=5)

    ax.axis('off')

def find_video(source):
    for search_dir in [REPO_ROOT / 'data' / 'lsa_raw' / 'videos', VIDEOS_DIR]:
        for ext in ['.MOV', '.mp4', '.avi', '.mkv']:
            p = search_dir / f'{source}{ext}'
            if p.exists():
                return p
    return None

def read_frame(video_path, frame_idx):
    cap = cv2.VideoCapture(str(video_path))
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ret, frame = cap.read()
    cap.release()
    return frame if ret else None

In [ ]:
video_path = find_video(ex['source'])
kps_list   = ex['keypoints']
n_frames   = ex['n_frames']

if video_path is None:
    print(f'Video no encontrado: {ex["source"]}  →  {VIDEOS_DIR}')
else:
    print(f'Video: {video_path.name}  ({n_frames} frames de keypoints)')

if HAS_WIDGETS and video_path:
    @widgets.interact(
        frame_idx=widgets.IntSlider(min=0, max=n_frames-1, step=1, value=0,
                                    description='Frame:', continuous_update=False,
                                    layout=widgets.Layout(width='600px'))
    )
    def show_frame(frame_idx):
        img = read_frame(video_path, frame_idx * SAMPLE_RATE)
        if img is None:
            print(f'No se pudo leer frame {frame_idx}')
            return
        fig, ax = plt.subplots(figsize=figsize_for_frame(img))
        draw_kps(ax, img, kps_list[frame_idx])
        ax.set_title(f'Frame {frame_idx}/{n_frames-1} — {ex["source"]}')
        plt.tight_layout()
        plt.show()

elif video_path:
    img = read_frame(video_path, 0)
    if img is not None:
        fig, ax = plt.subplots(figsize=figsize_for_frame(img))
        draw_kps(ax, img, kps_list[0])
        ax.set_title(f'Frame 0 — {ex["source"]}')
        plt.tight_layout()
        plt.show()

## Sección 2 — Cargar dataset completo
Si `toy_dataset.json` no existe todavía, cae a `toy_test.json` (1 entrada).

In [ ]:
if TOY_DATASET_PATH.exists():
    with open(TOY_DATASET_PATH) as f:
        dataset = json.load(f)
    print(f'toy_dataset.json cargado: {len(dataset)} entradas')
else:
    with open(TOY_TEST_PATH) as f:
        dataset = json.load(f)
    print(f'⚠  toy_dataset.json no existe. Usando toy_test.json ({len(dataset)} entrada).')
    print('   Para generarlo: python scripts/build_toy_dataset.py --sample-rate 2')

In [ ]:
rows = []
for e in dataset:
    m = e['metadata']
    rows.append({
        'source'        : e['source'][:35],
        'intent'        : m.get('intent', ''),
        'n_frames'      : e['n_frames'],
        'duration_s'    : m.get('duration_s', 0),
        'confidence_avg': m.get('confidence_avg', 0),
        'pose_pct'      : m.get('pose_pct', 0),
        'face_pct'      : m.get('face_pct', 0),
        'left_hand_pct' : m.get('left_hand_pct', 0),
        'right_hand_pct': m.get('right_hand_pct', 0),
        'word_count'    : m.get('word_count', 0),
    })

df = pd.DataFrame(rows)
df.head()

## Sección 3 — Análisis lingüístico
TTR, n-gramas y distribución de longitud de texto.

In [ ]:
def tokenize(text):
    return re.findall(r'\b\w+\b', text.lower())

texts = [e['text'] for e in dataset]
all_tokens = [t for text in texts for t in tokenize(text)]
all_types  = set(all_tokens)

ttr_global = len(all_types) / len(all_tokens) if all_tokens else 0

ttrs = []
for text in texts:
    toks = tokenize(text)
    ttrs.append(len(set(toks)) / len(toks) if toks else 0)

def ngrams(tokens, n):
    return [tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1)]

top_bigrams  = Counter(ngrams(all_tokens, 2)).most_common(10)
top_trigrams = Counter(ngrams(all_tokens, 3)).most_common(10)
word_counts  = [len(tokenize(t)) for t in texts]

print(f'Tokens totales  : {len(all_tokens)}')
print(f'Vocabulario único: {len(all_types)}')
print(f'TTR global      : {ttr_global:.3f}')

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# TTR por entrada
axes[0].bar(range(len(ttrs)), ttrs, color='steelblue', alpha=0.75)
axes[0].axhline(ttr_global, color='red', ls='--', label=f'Global: {ttr_global:.2f}')
axes[0].set_xlabel('Entrada'); axes[0].set_ylabel('TTR')
axes[0].set_title('TTR por entrada'); axes[0].legend()

# Top bigramas
bg_labels = [' '.join(bg) for bg, _ in top_bigrams]
bg_counts = [c for _, c in top_bigrams]
axes[1].barh(bg_labels[::-1], bg_counts[::-1], color='teal', alpha=0.75)
axes[1].set_title('Top 10 bigramas'); axes[1].set_xlabel('Frecuencia')

# Longitud de texto
axes[2].hist(word_counts, bins=max(5, len(word_counts)//2), color='coral', alpha=0.75, edgecolor='white')
axes[2].set_xlabel('Palabras por entrada'); axes[2].set_ylabel('Entradas')
axes[2].set_title('Distribución longitud de texto')

plt.tight_layout()
plt.show()

## Sección 4 — Distribución por intent

In [ ]:
intents = [e['metadata'].get('intent', 'sin_intent') for e in dataset]
intent_counts = Counter(intents)

labels  = [k for k, _ in intent_counts.most_common()]
counts  = [v for _, v in intent_counts.most_common()]
colors  = plt.cm.Set2(np.linspace(0, 1, len(labels)))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

bars = axes[0].bar(labels, counts, color=colors, alpha=0.85, edgecolor='white')
axes[0].set_ylabel('Videos'); axes[0].set_title('Distribución por intent')
axes[0].tick_params(axis='x', rotation=30)
for bar, c in zip(bars, counts):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                 str(c), ha='center', va='bottom', fontsize=10)

axes[1].pie(counts, labels=labels, colors=colors, autopct='%1.0f%%', startangle=90)
axes[1].set_title('Proporción por intent')

plt.tight_layout()
plt.show()

## Sección 5 — Calidad de keypoints
Tabla con gradiente de color (rojo = bajo, verde = alto) + boxplots por componente.

In [ ]:
quality_cols = ['confidence_avg', 'pose_pct', 'face_pct', 'left_hand_pct', 'right_hand_pct']

styled = (
    df.style
    .background_gradient(subset=quality_cols, cmap='RdYlGn', vmin=0, vmax=1)
    .format({
        'confidence_avg': '{:.2f}',
        'pose_pct'      : '{:.0%}',
        'face_pct'      : '{:.0%}',
        'left_hand_pct' : '{:.0%}',
        'right_hand_pct': '{:.0%}',
        'duration_s'    : '{:.1f}s',
    })
)
display(styled)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
labels_bp = ['Pose', 'Cara', 'Mano Izq', 'Mano Der', 'Conf avg']
data_bp   = [df[c].values for c in quality_cols]
bp = ax.boxplot(data_bp, labels=labels_bp, patch_artist=True)
colors_bp = ['#4CAF50', '#2196F3', '#FF9800', '#F44336', '#9C27B0']
for patch, color in zip(bp['boxes'], colors_bp):
    patch.set_facecolor(color); patch.set_alpha(0.6)
ax.set_ylim(0, 1.05)
ax.set_ylabel('Porcentaje / confianza')
ax.set_title('Calidad de detección por componente')
plt.tight_layout()
plt.show()

## Sección 6 — Simulacro de escala
¿Qué tan lejos estamos de LSA-T?

In [ ]:
LSA_T_EXAMPLES = 14_880
LSA_T_VOCAB    = 13_664

n_ours   = len(dataset)
our_vocab = len(all_types)
avg_dur  = df['duration_s'].mean() if 'duration_s' in df.columns else 30

# Curva de crecimiento de vocabulario
vocab_growth = []
seen = set()
for e in dataset:
    seen.update(tokenize(e['text']))
    vocab_growth.append(len(seen))

factor = LSA_T_EXAMPLES / n_ours
hours_needed = (LSA_T_EXAMPLES - n_ours) * avg_dur / 3600

print('=' * 52)
print('SIMULACRO DE ESCALA')
print('=' * 52)
print(f'  Toy dataset (actual) : {n_ours:>6} ejemplos | {our_vocab:>5} palabras únicas')
print(f'  LSA-T (referencia)   : {LSA_T_EXAMPLES:>6} ejemplos | {LSA_T_VOCAB:>5} palabras únicas')
print(f'  Factor de escala     : {factor:.0f}x')
print(f'  Videos adicionales   : ~{LSA_T_EXAMPLES - n_ours}')
print(f'  Horas adicionales    : ~{hours_needed:.1f}h (a ~{avg_dur:.0f}s/video)')
print(f'  Cobertura vocab      : {our_vocab/LSA_T_VOCAB*100:.1f}% ({our_vocab}/{LSA_T_VOCAB})')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(range(1, n_ours+1), vocab_growth, 'b-o', ms=4, label='Toy dataset')
axes[0].axhline(LSA_T_VOCAB, color='red', ls='--', alpha=0.6, label=f'LSA-T vocab ({LSA_T_VOCAB})')
axes[0].set_xlabel('Ejemplos'); axes[0].set_ylabel('Vocabulario único')
axes[0].set_title('Curva de crecimiento del vocabulario')
axes[0].legend()

bars2 = axes[1].bar(
    ['Toy dataset\n(actual)', 'LSA-T\n(objetivo)'],
    [n_ours, LSA_T_EXAMPLES],
    color=['steelblue', 'coral'], alpha=0.8, edgecolor='white'
)
axes[1].set_ylabel('Número de ejemplos')
axes[1].set_title('Escala: toy vs LSA-T')
for bar, v in zip(bars2, [n_ours, LSA_T_EXAMPLES]):
    axes[1].text(bar.get_x() + bar.get_width()/2, v + 100,
                 str(v), ha='center', va='bottom', fontsize=11)

plt.tight_layout()
plt.show()